# 11. Pipeline End-to-End Test

This notebook runs the full pipeline from raw pose CSV to digital biomarker output
using `pipeline.run_pipeline()` and `configs/pipeline_default.yaml`.

Pipeline order:

```
Validation → Annotation → Preprocessing → Normalization → Motion Attribution → Features → Biomech → Scoring
```

Only steps with `enabled: true` in the YAML config are executed.
Steps not yet implemented raise `NotImplementedError` when enabled.

Currently implemented steps (can be safely enabled):

- validation
- annotation
- normalization

This notebook assumes that all individual module tests (00 through 10) are passing.

> **Status:** End-to-end test covers only implemented steps.
> This notebook will be extended as each new module is completed.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import load_pipeline_config, run_pipeline

In [ ]:
config_path = Path("../configs/pipeline_default.yaml")
csv_path = "../data/sample/mediapipe_forward_bend_sample.csv"
ann_path = "../data/sample/mediapipe_forward_bend_sample_annotation.csv"

config = load_pipeline_config(config_path)
df = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

print("pipeline config loaded")
print("  validation enabled:   ", config.validation.enabled)
print("  annotation enabled:   ", config.annotation.enabled)
print("  preprocessing enabled:", config.preprocessing.enabled)
print("  normalization enabled:", config.normalization.enabled)
print("  motion_attribution:   ", config.motion_attribution.enabled)
print("  features enabled:     ", config.features.enabled)

In [ ]:
result_df, report = run_pipeline(df, config=config, landmarks=LANDMARKS, ann_df=ann_df)

print(f"output shape: {result_df.shape[0]} frames, {result_df.shape[1]} columns")
print()
print("steps executed:", list(report.keys()))

In [ ]:
for step, step_report in report.items():
    passed = step_report.get("passed", step_report.get("annotation_provided", "n/a"))
    print(f"  {step:20s}  {passed}")

## Interpretation

Expected results with default config (validation + annotation + normalization enabled):

- `steps executed` contains `['validation', 'annotation', 'normalization']`
- `validation` step passes
- `annotation` step applied
- `normalization` step applied
- output dataframe contains both raw and normalized coordinate columns
- output dataframe contains annotation metadata columns

To test additional steps, enable them in `configs/pipeline_default.yaml` once implemented.